# EOQ Lab — Colorado & Texas Public Education Data Collection

**States:** Colorado, Texas  
**Goal:** Download as much publicly available K–12 education data as practical into `data/raw/{state}/`, and cut **state-only rows** from existing national (federal) data into `data/cleaned/{state}/`.

**How this differs from Hawaii/Virginia:**
- Virginia used a **CKAN** API (`data.virginia.gov`).
- Colorado and Texas use **Socrata** open data APIs (`data.colorado.gov`, `data.texas.gov`) plus **BeautifulSoup** scraping of state DOE pages.

**Prerequisites:** Federal notebooks should already have run (NCES CCD school zip + CRDC extracts in `data/raw/federal/`). Phase 7 reuses those files.

**Run order:** **Step 0.2** → Phases 1–8, **one cell at a time** (Shift+Enter). This notebook does not auto-run.


---
## Step 0.1 — Packages *(run once in a terminal — not in this notebook)*

**Do not run `%pip` in this notebook** — it can hang the kernel on Windows and make the notebook feel "out of control."

In a **terminal** (project root), if needed:

```bash
py -3 -m pip install -r requirements.txt
```

Then open this notebook and start at **Step 0.2** below.


---
## Step 0.2 — Project paths & imports

Sets `PROJECT_ROOT`, creates `data/raw/colorado/`, `data/raw/texas/`, and matching `data/cleaned/` folders.

**Max-collection settings** (tune here before running later phases):


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

# Find project root (works when cwd is project root or notebooks/)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

from federal_collect import refresh_manifest
from state_collect import (
    STATE_ABBRS,
    STATE_SLUGS,
    discover_html_seed_pages,
    discover_socrata_datasets,
    direct_download_catalog,
    download_state_rows,
    ensure_state_layout,
    filter_crdc_extracted_for_state,
    filter_nces_ccd_for_state,
    filter_nces_f33_for_state,
    state_collection_summary,
)
import colorado_collect
import texas_collect

PATHS = ensure_state_layout(PROJECT_ROOT, STATE_SLUGS)
DOWNLOAD_LOG = PATHS["logs_dir"] / "download_log.jsonl"
MANIFEST_CSV = PATHS["logs_dir"] / "manifest.csv"

CO_DISCOVERED = PATHS["logs_dir"] / "co_discovered_links.csv"
TX_DISCOVERED = PATHS["logs_dir"] / "tx_discovered_links.csv"
TX_ARCGIS_DISCOVERED = PATHS["logs_dir"] / "tx_arcgis_discovered.csv"

# Pause between bulk downloads (seconds) — polite to state servers
SECONDS_BETWEEN_DOWNLOADS = 0.5

# Texas PEIMS single-file page lists 80+ ZIPs; 0 = download all years (very large)
MAX_TX_PEIMS_ZIP_DOWNLOADS = 0

# Socrata: skip map/story views that 404 on CSV export (recommended)
SOCRATA_EXPORTABLE_ONLY = True
SOCRATA_MAX_PER_QUERY = 200
SECONDS_BETWEEN_SOCRATA_CHECKS = 0.3

# Texas ArcGIS: download CSV layers after cataloging (Phase 6b)
DOWNLOAD_ARCGIS_CSV = True

print("Project root:", PROJECT_ROOT.resolve())
print("States:", list(STATE_SLUGS))
print("Download log:", DOWNLOAD_LOG.resolve())
print("Max mode: PEIMS cap =", MAX_TX_PEIMS_ZIP_DOWNLOADS, "| Socrata exportable only =", SOCRATA_EXPORTABLE_ONLY)


Project root: C:\Users\saihaj\Documents\Data Collection - EOQ Lab
States: ['colorado', 'texas']
Download log: C:\Users\saihaj\Documents\Data Collection - EOQ Lab\logs\download_log.jsonl
Max mode: PEIMS cap = 0 | Socrata exportable only = True


---
## Phase 1 — Direct downloads

Hand-picked stable CDE/TEA URLs before automated discovery. Files land in `data/raw/{state}/{category}/`.


In [2]:
phase1_catalog = pd.concat(
    [
        direct_download_catalog("colorado", colorado_collect.DIRECT_DOWNLOADS),
        direct_download_catalog("texas", texas_collect.DIRECT_DOWNLOADS),
    ],
    ignore_index=True,
)
print(f"Phase 1 URLs queued: {len(phase1_catalog)}")
phase1_catalog


Phase 1 URLs queued: 3


,state,source,dataset_title,description,category,url,dest_filename,format
0,colorado,Direct,CDE 2025 Preliminary District Performance Rati...,Colorado: CDE 2025 preliminary district rating...,test_scores,https://ed.cde.state.co.us/fs/resource-manager...,DPF2025_PreliminaryRatingsOverTime.xlsx,XLSX
1,colorado,Direct,CDE 2025 Preliminary School Performance Rating...,Colorado: CDE 2025 preliminary school ratings ...,test_scores,https://ed.cde.state.co.us/fs/resource-manager...,SPF2025_PreliminaryRatingsOverTime.xlsx,XLSX
2,texas,Direct,PEIMS Financial Data Dictionary,Texas: PEIMS financial data dictionary (docume...,financials,https://tea.texas.gov/sites/default/files/PEIM...,PEIMS-Financial-Data-Dictionary.pdf,PDF


In [3]:
phase1_records = download_state_rows(
    phase1_catalog,
    PROJECT_ROOT,
    DOWNLOAD_LOG,
    seconds_between=SECONDS_BETWEEN_DOWNLOADS,
)
print(
    f"downloaded: {sum(1 for r in phase1_records if r['status']=='downloaded')}, "
    f"skipped: {sum(1 for r in phase1_records if r['status']=='skipped_exists')}, "
    f"failed: {sum(1 for r in phase1_records if r['status']=='failed')}"
)


downloaded: 0, skipped: 2, failed: 1


---
## Phase 2 — Colorado open data (Socrata API)

**Portal:** [data.colorado.gov](https://data.colorado.gov)  
**Method:** Socrata catalog API. Tabular datasets export as CSV; map-only views are skipped when `SOCRATA_EXPORTABLE_ONLY` is True.

If you see **429 Too Many Requests**, wait 1–2 minutes, **restart the kernel**, re-run Step 0.2, and retry this cell (the script retries with backoff).

Discovered links are saved to `logs/co_discovered_links.csv` before download.


In [4]:
co_socrata_df = discover_socrata_datasets(
    colorado_collect.SOCRATA_BASE,
    colorado_collect.SOCRATA_SEARCH_QUERIES,
    "colorado",
    attribution_keywords=colorado_collect.SOCRATA_ATTRIBUTION_KEYWORDS,
    max_per_query=SOCRATA_MAX_PER_QUERY,
    exportable_only=SOCRATA_EXPORTABLE_ONLY,
    seconds_between_checks=SECONDS_BETWEEN_SOCRATA_CHECKS,
)
co_socrata_df.to_csv(CO_DISCOVERED, index=False)
print(f"Colorado Socrata datasets discovered (exportable): {len(co_socrata_df)}")
print("Saved:", CO_DISCOVERED)
co_socrata_df.head(10)


Colorado Socrata datasets discovered (exportable): 19
Saved: c:\Users\saihaj\Documents\Data Collection - EOQ Lab\logs\co_discovered_links.csv


,state,source,dataset_id,dataset_title,description,category,url,dest_filename,format
0,colorado,Socrata,cfyh-6xxg,District Graduation Data by Instructional Prog...,Colorado open data: District Graduation Data b...,enrollment,https://data.colorado.gov/api/views/cfyh-6xxg/...,socrata_cfyh-6xxg.csv,CSV
1,colorado,Socrata,9nuw-2h55,Graduation Data by School and Instructional Pr...,Colorado open data: Graduation Data by School ...,enrollment,https://data.colorado.gov/api/views/9nuw-2h55/...,socrata_9nuw-2h55.csv,CSV
2,colorado,Socrata,hgwg-pzjd,2012 TCAP School and District Summary Results ...,Colorado open data: 2012 TCAP School and Distr...,test_scores,https://data.colorado.gov/api/views/hgwg-pzjd/...,socrata_hgwg-pzjd.csv,CSV
3,colorado,Socrata,44v8-6fzj,CSAP School And District Summary Results 2010,Colorado open data: CSAP School And District S...,test_scores,https://data.colorado.gov/api/views/44v8-6fzj/...,socrata_44v8-6fzj.csv,CSV
4,colorado,Socrata,6vnq-az4b,Crimes in Colorado 1997 to 2015,Colorado open data: Crimes in Colorado 1997 to...,discipline,https://data.colorado.gov/api/views/6vnq-az4b/...,socrata_6vnq-az4b.csv,CSV
5,colorado,Socrata,cudh-kv54,2013 TCAP School and District Summary Results ...,Colorado open data: 2013 TCAP School and Distr...,test_scores,https://data.colorado.gov/api/views/cudh-kv54/...,socrata_cudh-kv54.csv,CSV
6,colorado,Socrata,y2pa-67y6,2013 TCAP School and District Summary Results ...,Colorado open data: 2013 TCAP School and Distr...,test_scores,https://data.colorado.gov/api/views/y2pa-67y6/...,socrata_y2pa-67y6.csv,CSV
7,colorado,Socrata,mv5d-uk2v,2012 TCAP School and District Summary Results ...,Colorado open data: 2012 TCAP School and Distr...,test_scores,https://data.colorado.gov/api/views/mv5d-uk2v/...,socrata_mv5d-uk2v.csv,CSV
8,colorado,Socrata,pxq3-yhfb,2012 TCAP School and District Summary Results ...,Colorado open data: 2012 TCAP School and Distr...,test_scores,https://data.colorado.gov/api/views/pxq3-yhfb/...,socrata_pxq3-yhfb.csv,CSV
9,colorado,Socrata,m5tx-hyc3,2013 TCAP School and District Summary Results ...,Colorado open data: 2013 TCAP School and Distr...,test_scores,https://data.colorado.gov/api/views/m5tx-hyc3/...,socrata_m5tx-hyc3.csv,CSV


In [5]:
co_socrata_records = download_state_rows(
    co_socrata_df,
    PROJECT_ROOT,
    DOWNLOAD_LOG,
    seconds_between=SECONDS_BETWEEN_DOWNLOADS,
)
print(
    f"CO Socrata — downloaded: {sum(1 for r in co_socrata_records if r['status']=='downloaded')}, "
    f"skipped: {sum(1 for r in co_socrata_records if r['status']=='skipped_exists')}, "
    f"failed: {sum(1 for r in co_socrata_records if r['status']=='failed')}"
)


CO Socrata — downloaded: 0, skipped: 19, failed: 0


---
## Phase 3 — Texas open data (Socrata API)

**Portal:** [data.texas.gov](https://data.texas.gov)  
Same Socrata pattern as Colorado, filtered to Texas Education Agency datasets.


In [6]:
tx_socrata_df = discover_socrata_datasets(
    texas_collect.SOCRATA_BASE,
    texas_collect.SOCRATA_SEARCH_QUERIES,
    "texas",
    attribution_keywords=texas_collect.SOCRATA_ATTRIBUTION_KEYWORDS,
    max_per_query=SOCRATA_MAX_PER_QUERY,
    exportable_only=SOCRATA_EXPORTABLE_ONLY,
    seconds_between_checks=SECONDS_BETWEEN_SOCRATA_CHECKS,
)
tx_socrata_df.to_csv(TX_DISCOVERED, index=False)
print(f"Texas Socrata datasets discovered (exportable): {len(tx_socrata_df)}")
tx_socrata_df.head(10)


Texas Socrata datasets discovered (exportable): 3


,state,source,dataset_id,dataset_title,description,category,url,dest_filename,format
0,texas,Socrata,6dh5-cse4,Superintendent Salary Reports by District for ...,Texas open data: Superintendent Salary Reports...,financials,https://data.texas.gov/api/views/6dh5-cse4/row...,socrata_6dh5-cse4.csv,CSV
1,texas,Socrata,nui6-x374,School Year 2024-2025 Statewide Accountability...,Texas open data: School Year 2024-2025 Statewi...,test_scores,https://data.texas.gov/api/views/nui6-x374/row...,socrata_nui6-x374.csv,CSV
2,texas,Socrata,hzek-udky,"AskTED Data - May 12, 2026","Texas open data: AskTED Data - May 12, 2026",test_scores,https://data.texas.gov/api/views/hzek-udky/row...,socrata_hzek-udky.csv,CSV


In [7]:
tx_socrata_records = download_state_rows(
    tx_socrata_df,
    PROJECT_ROOT,
    DOWNLOAD_LOG,
    seconds_between=SECONDS_BETWEEN_DOWNLOADS,
)
print(
    f"TX Socrata — downloaded: {sum(1 for r in tx_socrata_records if r['status']=='downloaded')}, "
    f"skipped: {sum(1 for r in tx_socrata_records if r['status']=='skipped_exists')}, "
    f"failed: {sum(1 for r in tx_socrata_records if r['status']=='failed')}"
)


TX Socrata — downloaded: 0, skipped: 3, failed: 0


---
## Phase 4 — Colorado CDE pages (BeautifulSoup)

Scrapes **direct file links** from Colorado Department of Education index pages (accountability, CMAS, PSAT, etc.).

CDE uses `/fs/resource-manager/view/{uuid}` links with a `data-file-name` attribute.


In [8]:
co_html_df = discover_html_seed_pages(
    colorado_collect.HTML_SEED_URLS,
    "colorado",
    page_label="CDE",
)
print(f"Colorado HTML links discovered: {len(co_html_df)}")
co_html_df.head(10)


Colorado HTML links discovered: 37


,state,source,dataset_title,description,category,url,dest_filename,format
0,colorado,HTML,DPF2025_PreliminaryRatingsOverTime_wAECs_11102...,Colorado CDE: DPF2025_PreliminaryRatingsOverTi...,other,https://ed.cde.state.co.us/fs/resource-manager...,bdbfed5c-32a6-43c1-a55d-cbc0c96c172d,
1,colorado,HTML,SPF2025_PreliminaryRatingsOverTime_wAECs_09112...,Colorado CDE: SPF2025_PreliminaryRatingsOverTi...,other,https://ed.cde.state.co.us/fs/resource-manager...,05ef35b0-dfd6-47fd-99af-2933bb35214b,
2,colorado,HTML,ADA_Achievement_w_Percentile_2025.xlsx,Colorado CDE: ADA_Achievement_w_Percentile_202...,other,https://ed.cde.state.co.us/fs/resource-manager...,c739a2ad-17f7-48fb-a7ec-cd339c27de48,
3,colorado,HTML,2025_CMASGrowthSummary-StateLevel_forMediaEmba...,Colorado CDE: 2025_CMASGrowthSummary-StateLeve...,test_scores,https://ed.cde.state.co.us/fs/resource-manager...,05e6cd9d-efd1-4e43-bb8e-a532ecddfa0b,
4,colorado,HTML,2025_PSATSATGrowthSummary-StateLevel_forMediaE...,Colorado CDE: 2025_PSATSATGrowthSummary-StateL...,test_scores,https://ed.cde.state.co.us/fs/resource-manager...,e67ac55b-809d-4763-b41b-bafbd0992edf,
5,colorado,HTML,2025_PSATSATGrowthSummary-StateDistrictSchool_...,Colorado CDE: 2025_PSATSATGrowthSummary-StateD...,test_scores,https://ed.cde.state.co.us/fs/resource-manager...,a759d797-7854-4888-b8ce-690033a97037,
6,colorado,HTML,2025_PSATSATGrowthSummary-StateDistrictSchool_...,Colorado CDE: 2025_PSATSATGrowthSummary-StateD...,test_scores,https://ed.cde.state.co.us/fs/resource-manager...,5fdbc6d2-e9b4-4793-af43-81eeb10a373f,
7,colorado,HTML,2025_WIDAACCESSGrowthSummary-StateDistrictScho...,Colorado CDE: 2025_WIDAACCESSGrowthSummary-Sta...,other,https://ed.cde.state.co.us/fs/resource-manager...,0eb524bf-635c-4026-9d05-86b9a1523f96,
8,colorado,HTML,2024_Matriculation-StateDistrictSchool_forSBEr...,Colorado CDE: 2024_Matriculation-StateDistrict...,other,https://ed.cde.state.co.us/fs/resource-manager...,e3b7132d-130b-436f-99c5-a84f712eb6dc,
9,colorado,HTML,2019to2025_ParticipationRate-StateDistrictScho...,Colorado CDE: 2019to2025_ParticipationRate-Sta...,other,https://ed.cde.state.co.us/fs/resource-manager...,56ba1da1-89ff-4556-ba8d-2666b88d1b83,


In [9]:
co_html_records = download_state_rows(
    co_html_df,
    PROJECT_ROOT,
    DOWNLOAD_LOG,
    seconds_between=SECONDS_BETWEEN_DOWNLOADS,
)
print(
    f"CO HTML — downloaded: {sum(1 for r in co_html_records if r['status']=='downloaded')}, "
    f"skipped: {sum(1 for r in co_html_records if r['status']=='skipped_exists')}, "
    f"failed: {sum(1 for r in co_html_records if r['status']=='failed')}"
)


CO HTML — downloaded: 0, skipped: 11, failed: 26


---
## Phase 5 — Texas TEA pages (BeautifulSoup)

Scrapes file links from Texas Education Agency report hubs (PEIMS finance, assessments, SAT/ACT, graduation, educator data, etc.).

**Note:** `MAX_TX_PEIMS_ZIP_DOWNLOADS = 0` downloads **every** PEIMS single-file ZIP on that page (many GB). Set to e.g. `6` for only the newest six years.


In [10]:
tx_html_df = discover_html_seed_pages(
    texas_collect.HTML_SEED_URLS,
    "texas",
    page_label="TEA",
)
before = len(tx_html_df)
tx_html_df = texas_collect.limit_texas_peims_downloads(tx_html_df, MAX_TX_PEIMS_ZIP_DOWNLOADS)
print(f"Texas HTML links discovered: {before} (downloading {len(tx_html_df)} after PEIMS cap)")
tx_html_df.head(10)


Texas HTML links discovered: 156 (downloading 156 after PEIMS cap)


,state,source,dataset_title,description,category,url,dest_filename,format
0,texas,HTML,Click here to access the data dictionary for t...,Texas TEA: Click here to access the data dicti...,financials,https://tea.texas.gov/data-reports/financial-r...,datadictionary.pdf,PDF
1,texas,HTML,District 2025-2026 Financial Budget Data (.zip...,Texas TEA: District 2025-2026 Financial Budget...,financials,https://tea.texas.gov/data-reports/financial-r...,budget2026.zip,ZIP
2,texas,HTML,Charter 2025-2026 Financial Budget Data (.zip ...,Texas TEA: Charter 2025-2026 Financial Budget ...,financials,https://tea.texas.gov/about-tea/state-funding/...,charbud26.zip,ZIP
3,texas,HTML,District 2024-2025 Financial Budget Data (.zip...,Texas TEA: District 2024-2025 Financial Budget...,financials,https://tea.texas.gov/data-reports/financial-r...,budget2025.zip,ZIP
4,texas,HTML,Charter 2024-2025 Financial Budget Data (.zip ...,Texas TEA: Charter 2024-2025 Financial Budget ...,financials,https://tea.texas.gov/data-reports/financial-r...,charbud25.zip,ZIP
5,texas,HTML,District 2023-2024 Financial Budget Data (.zip...,Texas TEA: District 2023-2024 Financial Budget...,financials,https://tea.texas.gov/data-reports/financial-r...,budget2024.zip,ZIP
6,texas,HTML,Charter 2023-2024 Financial Budget Data (.zip ...,Texas TEA: Charter 2023-2024 Financial Budget ...,financials,https://tea.texas.gov/data-reports/financial-r...,charbud24.zip,ZIP
7,texas,HTML,District 2023-2024 Financial Actual Data (.zip...,Texas TEA: District 2023-2024 Financial Actual...,test_scores,https://tea.texas.gov/data-reports/financial-r...,actget2024.zip,ZIP
8,texas,HTML,Charter 2023-2024 Financial Actual Data (.zip ...,Texas TEA: Charter 2023-2024 Financial Actual ...,test_scores,https://tea.texas.gov/data-reports/financial-r...,charact24.zip,ZIP
9,texas,HTML,District 2022-2023 Financial Budget Data (.zip...,Texas TEA: District 2022-2023 Financial Budget...,financials,https://tea.texas.gov/about-tea/state-funding/...,budget2023.zip,ZIP


In [11]:
tx_html_records = download_state_rows(
    tx_html_df,
    PROJECT_ROOT,
    DOWNLOAD_LOG,
    seconds_between=SECONDS_BETWEEN_DOWNLOADS,
)
print(
    f"TX HTML — downloaded: {sum(1 for r in tx_html_records if r['status']=='downloaded')}, "
    f"skipped: {sum(1 for r in tx_html_records if r['status']=='skipped_exists')}, "
    f"failed: {sum(1 for r in tx_html_records if r['status']=='failed')}"
)


TX HTML — downloaded: 52, skipped: 104, failed: 0


---
## Phase 6a — Texas ArcGIS open data (catalog)

**Portal:** [TEA Public Open Data on ArcGIS](https://schoolsdata2-tea-texas.opendata.arcgis.com/)

Catalogs datasets via the ArcGIS Search API → `logs/tx_arcgis_discovered.csv`.


In [12]:
tx_arcgis_df = texas_collect.discover_tea_arcgis_items(max_items=200)
tx_arcgis_df.to_csv(TX_ARCGIS_DISCOVERED, index=False)
print(f"Texas ArcGIS items cataloged: {len(tx_arcgis_df)}")
tx_arcgis_df.head(10)


Texas ArcGIS items cataloged: 43


,state,source,dataset_id,dataset_title,description,category,url,dest_filename,format
0,texas,ArcGIS,a015ac7bc44c4d738df9b6ecb4c845f2,All programs of study Polygons 2,Texas TEA ArcGIS open data: All programs of st...,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV Collection
1,texas,ArcGIS,9b0984f830bf4808852103121b15dd55,CTE ESC Regions,Texas TEA ArcGIS open data: CTE ESC Regions,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV
2,texas,ArcGIS,87c957f2ec334c83bc2b952bf6e64344,DistCounty,Texas TEA ArcGIS open data: DistCounty,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV
3,texas,ArcGIS,aa8baba0a0bd444a9b620e84a6d796cf,AllPOS_5S4Campus,Texas TEA ArcGIS open data: AllPOS_5S4Campus,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV
4,texas,ArcGIS,762dc5ff24554ea49cebb6bce21c0298,AdjacentDistrict,Texas TEA ArcGIS open data: AdjacentDistrict,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV
5,texas,ArcGIS,5a98daef9e434fd889544db64c7ee78e,School2012,Texas TEA ArcGIS open data: School2012,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,Shapefile
6,texas,ArcGIS,354d55c936f94957be3218bbae9fbeb7,AllPOS_5S420_21_District,Texas TEA ArcGIS open data: AllPOS_5S420_21_Di...,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV
7,texas,ArcGIS,85092ce228df4307be9e04c03f26e2fb,CTE AllPOS_5S420_21_District,Texas TEA ArcGIS open data: CTE AllPOS_5S420_2...,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV
8,texas,ArcGIS,6d2419b050e64ca3b79a363632b1ebfd,CountyDist,Texas TEA ArcGIS open data: CountyDist,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV
9,texas,ArcGIS,47a8009b308c4d5fa54ebf03dd9e1d3d,SD_Approx_Area,Texas TEA ArcGIS open data: SD_Approx_Area,other,https://schoolsdata2-tea-texas.opendata.arcgis...,,CSV


---
## Phase 6b — Texas ArcGIS CSV downloads

Resolves ArcGIS Online item metadata and downloads CSV layers via the `/data` endpoint (Hub bulk exports often fail).


In [13]:
if DOWNLOAD_ARCGIS_CSV:
    tx_arcgis_catalog = texas_collect.build_arcgis_download_catalog(tx_arcgis_df)
    print(f"ArcGIS CSV layers queued: {len(tx_arcgis_catalog)}")
    tx_arcgis_records = download_state_rows(
        tx_arcgis_catalog,
        PROJECT_ROOT,
        DOWNLOAD_LOG,
        seconds_between=SECONDS_BETWEEN_DOWNLOADS,
    )
    print(
        f"TX ArcGIS — downloaded: {sum(1 for r in tx_arcgis_records if r['status']=='downloaded')}, "
        f"skipped: {sum(1 for r in tx_arcgis_records if r['status']=='skipped_exists')}, "
        f"failed: {sum(1 for r in tx_arcgis_records if r['status']=='failed')}"
    )
else:
    print("Skipped ArcGIS downloads (DOWNLOAD_ARCGIS_CSV = False)")


ArcGIS CSV layers queued: 9
TX ArcGIS — downloaded: 9, skipped: 0, failed: 0


---
## Phase 7 — Federal data filtered to Colorado & Texas

Reuses **national files already on disk** (no re-download of full CRDC zips):

1. **NCES CCD** school universe → filter `ST == 'CO'` or `'TX'` → `data/cleaned/{state}/enrollment/`
2. **NCES F-33** district finance (if `Sdf*_1a.zip` on disk) → `data/cleaned/{state}/financials/`
3. **CRDC extracted CSVs** → filter `LEA_STATE` → `data/cleaned/{state}/{category}/`

Phase 7 may take several minutes for Texas (large row counts).


In [14]:
federal_filter_summary = []

for state_slug in STATE_SLUGS:
    abbr = STATE_ABBRS[state_slug]
    print(f"\n=== {state_slug.title()} ({abbr}) federal filter ===")

    nces_result = filter_nces_ccd_for_state(PROJECT_ROOT, state_slug, abbr, DOWNLOAD_LOG)
    print("NCES CCD schools:", nces_result)

    f33_result = filter_nces_f33_for_state(PROJECT_ROOT, state_slug, abbr, DOWNLOAD_LOG)
    print("NCES F-33 districts:", f33_result)

    crdc_written = filter_crdc_extracted_for_state(PROJECT_ROOT, state_slug, abbr, DOWNLOAD_LOG)
    n_written = sum(1 for r in crdc_written if r.get("status") == "written")
    n_skip = sum(1 for r in crdc_written if r.get("status") == "skipped_exists")
    print(f"CRDC extracts: {n_written} written, {n_skip} skipped (already on disk)")

    federal_filter_summary.append(
        {"state": state_slug, "nces": nces_result, "f33": f33_result, "crdc_files": n_written}
    )

pd.DataFrame(federal_filter_summary)



=== Colorado (CO) federal filter ===
NCES CCD schools: {'status': 'skipped_exists', 'local_path': 'c:\\Users\\saihaj\\Documents\\Data Collection - EOQ Lab\\data\\cleaned\\colorado\\enrollment\\nces_public_schools_CO.csv', 'rows': 0}
NCES F-33 districts: {'status': 'skipped_exists', 'local_path': 'c:\\Users\\saihaj\\Documents\\Data Collection - EOQ Lab\\data\\cleaned\\colorado\\financials\\nces_district_finance_CO.csv', 'rows': 0}
CRDC extracts: 0 written, 86 skipped (already on disk)

=== Texas (TX) federal filter ===
NCES CCD schools: {'status': 'skipped_exists', 'local_path': 'c:\\Users\\saihaj\\Documents\\Data Collection - EOQ Lab\\data\\cleaned\\texas\\enrollment\\nces_public_schools_TX.csv', 'rows': 0}
NCES F-33 districts: {'status': 'written', 'local_path': 'c:\\Users\\saihaj\\Documents\\Data Collection - EOQ Lab\\data\\cleaned\\texas\\financials\\nces_district_finance_TX.csv', 'rows': 1237}
CRDC extracts: 0 written, 85 skipped (already on disk)


,state,nces,f33,crdc_files
0,colorado,"{'status': 'skipped_exists', 'local_path': 'c:...","{'status': 'skipped_exists', 'local_path': 'c:...",0
1,texas,"{'status': 'skipped_exists', 'local_path': 'c:...","{'status': 'written', 'local_path': 'c:\Users\...",0


---
## Phase 8 — Manifest refresh & summary

Rebuilds `logs/manifest.csv` and prints file counts per state (raw + cleaned).


In [15]:
manifest = refresh_manifest(DOWNLOAD_LOG, MANIFEST_CSV)

for state_slug in STATE_SLUGS:
    print(f"\n=== {state_slug.upper()} collection summary ===")
    summary = state_collection_summary(PROJECT_ROOT, state_slug)
    if summary.empty:
        print("(no files yet)")
    else:
        print(summary.to_string(index=False))

state_rows = manifest[manifest["state"].isin(STATE_SLUGS)] if not manifest.empty else pd.DataFrame()
if not state_rows.empty:
    print("\nManifest rows by state:")
    print(state_rows.groupby(["state", "category"]).size())

print(f"\nManifest: {MANIFEST_CSV}")
print("Regenerate docs: python scripts/generate_docs.py")



=== COLORADO collection summary ===
  layer    category  files
    raw test_scores     20
    raw  enrollment      4
    raw  discipline      1
    raw       other      7
    raw       TOTAL     32
cleaned test_scores     16
cleaned  enrollment     20
cleaned  financials      4
cleaned  discipline     10
cleaned       other     38
cleaned       TOTAL     88

=== TEXAS collection summary ===
  layer    category  files
    raw test_scores    124
    raw  enrollment      1
    raw  financials     33
    raw    teachers      1
    raw       other      9
    raw       TOTAL    168
cleaned test_scores     16
cleaned  enrollment     19
cleaned  financials      4
cleaned  discipline     10
cleaned       other     38
cleaned       TOTAL     87

Manifest rows by state:
state     category   
colorado  discipline      16
          enrollment      78
          financials       7
          other          165
          teachers        24
          test_scores    133
texas     discipline      10
    